# YOLO26 + RepViT-M0.9

沿用 EdgeViT / EfficientViT 的最小集成：官方 RepViT 作为骨干，`Index` 取三个尺度，再直连 YOLO26 的 `Detect`（不包含原版 YOLO26 的 PAN/FPN neck）。固定 Ultralytics `v8.4.0`、RepViT 提交和 timm `1.0.24`。A/B 段保留随机初始化基线；C 段可选择是否用 ImageNet 权重初始化 RepViT 骨干，默认不加载。

1. **A 段**：在新 Colab 会话执行初始化、基线检查、生成适配器和注册补丁。
2. **B 段**：先**重新启动会话**（不要断开并删除运行时），再从 B1 继续。若 VM 已重置，从 A 段重做。
3. **C 段**：可选 COCO8 单轮训练，只验证训练链路，不用于比较精度。


In [1]:
# A1. 拉取并安装固定版本；专用目录不与其他 Notebook 的 /content/ultralytics 共用。
from pathlib import Path
import subprocess
import sys

WORK = Path('/content/yolo26_repvit_work')
ULTRA = WORK / 'ultralytics'
REP = WORK / 'RepViT'
ULTRA_REF = 'v8.4.0'
REP_REF = '298f42075eda5d2e6102559fad260c970769d34e'
ULTRA_URL = 'https://github.com/ultralytics/ultralytics.git'
REP_URL = 'https://github.com/THU-MIG/RepViT.git'

def capture(repo, *args):
    return subprocess.check_output(['git', *args], cwd=repo, text=True).strip()

def checkout(repo, url, ref):
    if not repo.exists():
        repo.mkdir(parents=True)
        subprocess.run(['git', 'init', str(repo)], check=True)
        subprocess.run(['git', 'remote', 'add', 'origin', url], cwd=repo, check=True)
    if not (repo / '.git').is_dir() or capture(repo, 'remote', 'get-url', 'origin') != url:
        raise RuntimeError(f'不覆盖来源不明的目录：{repo}')
    local_ref = f'refs/tags/{ref}^{{commit}}' if ref.startswith('v') else ref
    head = subprocess.run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=repo, capture_output=True, text=True)
    wanted = subprocess.run(['git', 'rev-parse', '--verify', local_ref], cwd=repo, capture_output=True, text=True)
    if head.returncode == wanted.returncode == 0 and head.stdout == wanted.stdout:
        return
    if capture(repo, 'status', '--porcelain'):
        raise RuntimeError(f'不切换包含本地改动的目录：{repo}')
    fetch_ref = f'refs/tags/{ref}:refs/tags/{ref}' if ref.startswith('v') else ref
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', fetch_ref], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=repo, check=True)
    if capture(repo, 'rev-parse', 'HEAD') != capture(repo, 'rev-parse', local_ref):
        raise RuntimeError(f'检出版本与预期不匹配：{repo}')

checkout(ULTRA, ULTRA_URL, ULTRA_REF)
checkout(REP, REP_URL, REP_REF)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ULTRA), 'timm==1.0.24'], check=True)
print('Ultralytics:', capture(ULTRA, 'rev-parse', 'HEAD'))
print('RepViT:', capture(REP, 'rev-parse', 'HEAD'))


Ultralytics: f2d3aed634a5b0e4828024718d4a61ab2f83fb19
RepViT: 298f42075eda5d2e6102559fad260c970769d34e


In [2]:
# A2. 只使用仓库内的原版 YOLO26 YAML；不下载权重。
import os
import torch

if 'ultralytics' in sys.modules:
    raise RuntimeError('请在新会话执行 A 段；当前 Python 已导入 Ultralytics。')
os.chdir(ULTRA)
sys.path.insert(0, str(ULTRA))
import ultralytics
from ultralytics import YOLO

assert Path(ultralytics.__file__).resolve().is_relative_to(ULTRA.resolve())
baseline = YOLO(str(ULTRA / 'ultralytics/cfg/models/26/yolo26.yaml'), task='detect')
assert baseline.model.stride.tolist() == [8.0, 16.0, 32.0]
print('baseline OK；PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
del baseline


WARNING ⚠️ no model scale passed. Assuming scale='n'.
baseline OK；PyTorch: 2.14.1+cu130 CUDA: True


In [3]:
# A3. 准备适配器、官方源码兼容改动和检测 YAML。此格不写文件。
ADAPTER = '''# Ultralytics AGPL-3.0 License - https://ultralytics.com/license
import torch
from torch import nn


class RepViT_M0_9(nn.Module):
    # RepViT-M0.9 stage-end [P3, P4, P5], strides [8, 16, 32].
    out_channels = (96, 192, 384)

    def __init__(self):
        super().__init__()
        # Delay loading timm/RepViT until the custom backbone is constructed.
        from .repvit_official import repvit_m0_9, _make_divisible

        model = repvit_m0_9(pretrained=False, num_classes=0, distillation=False)
        self.features = model.features  # omit classification head and global pooling
        stride = 4  # two stride-2 convolutions in features[0]
        stage_ends = {}
        for index, (_, _, channels, _, _, step) in enumerate(model.cfgs, start=1):
            stride *= step
            stage_ends[stride] = (index, _make_divisible(channels, 8))
        stages = tuple(stage_ends[s] for s in (8, 16, 32))
        self.out_indices = tuple(index for index, _ in stages)
        if tuple(channels for _, channels in stages) != self.out_channels:
            raise ValueError("RepViT-M0.9 stage channels do not match the YOLO26 YAML")

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        outputs = []
        for index, layer in enumerate(self.features):
            x = layer(x)
            if index in self.out_indices:
                outputs.append(x)
        return outputs
'''

CFG = '''# YOLO26 + RepViT-M0.9 (direct detection head, no PAN/FPN).
nc: 80
end2end: true
reg_max: 1

backbone:
  - [-1, 1, RepViT_M0_9, []]

head:
  - [0, 1, Index, [96, 0]]    # 1: P3 / 8
  - [0, 1, Index, [192, 1]]   # 2: P4 / 16
  - [0, 1, Index, [384, 2]]   # 3: P5 / 32
  - [[1, 2, 3], 1, Detect, [nc]]
'''

# 源码副本放在包内，确保其模块路径可被 checkpoint 重新导入。
# 使用新版 timm 的公开导入，移除副本的全局模型注册，避免覆盖 timm 同名实现。
def replace_once(text, old, new):
    if text.count(old) != 1:
        raise RuntimeError(f'预期恰有一个补丁锚点：{old!r}')
    return text.replace(old, new, 1)

VENDOR = (REP / 'model/repvit.py').read_text(encoding='utf-8')
VENDOR = replace_once(VENDOR, 'from timm.models.layers import SqueezeExcite', 'from timm.layers import SqueezeExcite')
VENDOR = replace_once(VENDOR, 'from timm.models.vision_transformer import trunc_normal_', 'from timm.layers import trunc_normal_')
VENDOR = replace_once(VENDOR, 'from timm.models import register_model\n', '')
if VENDOR.count('@register_model\n') != 6:
    raise RuntimeError('RepViT 注册装饰器数量与固定版本不符。')
VENDOR = '# Adapted from THU-MIG/RepViT; see REPViT-LICENSE.\n' + VENDOR.replace('@register_model\n', '')


In [4]:
# A4. 注册模块、补 parse_model，然后写入适配器与 YAML。
import ast

INIT_PATH = 'ultralytics/nn/modules/__init__.py'
TASKS_PATH = 'ultralytics/nn/tasks.py'
EXTRA = ULTRA / 'ultralytics/nn/modules/extramodules'
CFG_PATH = ULTRA / 'ultralytics/cfg/models/26/yolo26_repvit.yaml'

def original(relative_path):
    return subprocess.check_output(['git', 'show', f'HEAD:{relative_path}'], cwd=ULTRA).decode('utf-8')

old_init = original(INIT_PATH)
fixed_init = replace_once(old_init, '__all__ = (', 'from .extramodules.repvit import RepViT_M0_9\n\n__all__ = (\n    "RepViT_M0_9",')
old_tasks = original(TASKS_PATH)
fixed_tasks = replace_once(old_tasks, '    RepVGGDW,\n', '    RepVGGDW,\n    RepViT_M0_9,\n')
fixed_tasks = replace_once(fixed_tasks, '        if m in base_modules:', '''        if m is RepViT_M0_9:
            if i != 0 or f != -1 or n_ != 1 or args or ch[f] != 3:
                raise ValueError("RepViT_M0_9 must be the first RGB layer with repeats=1 and no arguments")
            c2 = list(m.out_channels)
        elif m in base_modules:''')
fixed_tasks = replace_once(fixed_tasks, '        elif m in frozenset({TorchVision, Index}):\n            c2 = args[0]\n            c1 = ch[f]\n            args = [*args[1:]]', '''        elif m in frozenset({TorchVision, Index}):
            c2 = args[0]
            c1 = ch[f]
            if m is Index and isinstance(c1, list):
                if len(args) != 2 or not 0 <= args[1] < len(c1) or c2 != c1[args[1]]:
                    raise ValueError(f"Index layer {i}: mismatched index/channels {args} for {c1}")
            args = [*args[1:]]''')

for label, source in [('modules', fixed_init), ('tasks', fixed_tasks), ('adapter', ADAPTER), ('repvit', VENDOR)]:
    ast.parse(source, filename=label)

# 先检查所有目标，再写入；允许原版和本 Notebook 的相同输出，不覆盖其他修改。
changes = {
    ULTRA / INIT_PATH: (old_init, fixed_init),
    ULTRA / TASKS_PATH: (old_tasks, fixed_tasks),
    EXTRA / '__init__.py': (None, ''),
    EXTRA / 'repvit.py': (None, ADAPTER),
    EXTRA / 'repvit_official.py': (None, VENDOR),
    EXTRA / 'REPViT-LICENSE': (None, (REP / 'LICENSE').read_text(encoding='utf-8')),
    CFG_PATH: (None, CFG),
}
for file, (before, after) in changes.items():
    current = file.read_text(encoding='utf-8') if file.exists() else None
    if current not in (before, after):
        raise RuntimeError(f'拒绝覆盖已有改动：{file}')
for file, (_, after) in changes.items():
    if file.exists() and file.read_text(encoding='utf-8') == after:
        continue
    file.parent.mkdir(parents=True, exist_ok=True)
    file.write_text(after, encoding='utf-8')
print('补丁已写入；请重新启动会话，再从 B1 开始。', CFG_PATH)


补丁已写入；请重新启动会话，再从 B1 开始。 /content/yolo26_repvit_work/ultralytics/ultralytics/cfg/models/26/yolo26_repvit.yaml


## 重新启动会话，再执行 B 段

A2 导入过 Ultralytics，写入补丁后 Python 不会自动刷新旧模块。重启后保留同一 VM，不要重新执行 A2。骨干可直接处理 stride 32 对齐的矩形输入，无需插值。在新 VM 加载自定义 checkpoint 前，也需要先完成 A 段并重启。


In [1]:
# B1. 检查独立适配器，再构建完整模型。
import os
from pathlib import Path
import sys

ULTRA = Path('/content/yolo26_repvit_work/ultralytics')
CFG_PATH = ULTRA / 'ultralytics/cfg/models/26/yolo26_repvit.yaml'
if not CFG_PATH.is_file():
    raise RuntimeError('没有找到生成的 YAML；请在当前 VM 先完成 A 段。')
if 'ultralytics' in sys.modules:
    raise RuntimeError('请重新启动会话，再从 B1 执行。')
os.chdir(ULTRA)
sys.path.insert(0, str(ULTRA))
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.nn.modules import RepViT_M0_9

assert Path(ultralytics.__file__).resolve().is_relative_to(ULTRA.resolve())
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
backbone = RepViT_M0_9().to(DEVICE).eval()
with torch.inference_mode():
    for h, w in [(256, 256), (384, 640)]:
        features = backbone(torch.zeros(1, 3, h, w, device=DEVICE))
        shapes = [tuple(x.shape) for x in features]
        expected = [(1, c, h // s, w // s) for c, s in zip((96, 192, 384), (8, 16, 32))]
        assert shapes == expected, (shapes, expected)
        print(f'{h}x{w} -> {shapes}')
del features, backbone

model = YOLO(str(CFG_PATH), task='detect')
assert model.model.stride.tolist() == [8.0, 16.0, 32.0]
assert model.model.model[-1].end2end and model.model.model[-1].reg_max == 1
model.info(verbose=True)


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


256x256 -> [(1, 96, 32, 32), (1, 192, 16, 16), (1, 384, 8, 8)]
384x640 -> [(1, 96, 48, 80), (1, 192, 24, 40), (1, 384, 12, 20)]
YOLO26_repvit summary: 372 layers, 6,748,120 parameters, 6,748,120 gradients, 21.9 GFLOPs


(372, 6748120, 6748120, 21.8800128)

In [2]:
# B2. 前向：检查 postprocess 输出，以及三个尺度的预测位置总数。
net = model.model.to(DEVICE).eval()
h, w = 384, 640
with torch.inference_mode():
    detections, raw = net(torch.rand(1, 3, h, w, device=DEVICE))
    positions = sum((h // s) * (w // s) for s in (8, 16, 32))
    assert set(raw) == {'one2many', 'one2one'}
    for branch in raw.values():
        assert branch['boxes'].shape == (1, 4, positions)
        assert branch['scores'].shape == (1, net.model[-1].nc, positions)
    assert detections.ndim == 3 and detections.shape[0] == 1 and detections.shape[-1] == 6
    assert torch.isfinite(detections).all().item()
print('forward OK:', tuple(detections.shape), 'positions:', positions)


forward OK: (1, 300, 6) positions: 5040


In [3]:
# B3. 从 one2many 检测预测回传到骨干（one2one 的骨干特征在 v8.4.0 中被 detach）。
# 用代理目标验证梯度连通性；实际检测损失由可选的冒烟训练验证。
net = model.model.to(DEVICE).train()
net.zero_grad(set_to_none=True)
raw = net(torch.rand(1, 3, 256, 256, device=DEVICE))['one2many']
loss = raw['scores'].float().square().mean() + raw['boxes'].float().square().mean()
assert torch.isfinite(loss).item()
loss.backward()
first_conv = net.model[0].features[0][0].c
assert first_conv.weight.grad is not None
assert torch.isfinite(first_conv.weight.grad).all().item()
assert first_conv.weight.grad.abs().max().item() > 0
print('backward OK；RepViT stem 梯度非零且有限。')
net.zero_grad(set_to_none=True)
del raw, loss


backward OK；RepViT stem 梯度非零且有限。


## C 段（可选）：选择初始化方式，运行 COCO8 一轮冒烟训练

先执行 C0 选模式，再执行 C1。两种模式都会从头训练检测任务，不恢复优化器或上次训练进度；两者的检测头都随机初始化。COCO8 可能会自动下载。正式训练时请换成自己的检测数据 YAML 并调整训练参数。

若 C1 下方仍显示 `repvit_m0_9_smoke` 的旧输出，那是此前保存的**随机初始化**测试记录，不是 ImageNet 模式的测试结果。重新执行 C1 后，Colab 会替换该输出。

### C0. 选择骨干初始化方式

- `False`（默认）：从 YAML 随机初始化，不下载 ImageNet 权重。
- `True`：下载 THU-MIG 官方 RepViT-M0.9 ImageNet-1K 450e 蒸馏分类权重，只迁移 `features.*`，跳过分类头和蒸馏头。

切换后重新执行 **C0 → C1**；不用重跑 A/B 段，但当前会话需要已经执行过 B1。

In [5]:
# @title C0. 初始化设置
USE_IMAGENET_PRETRAINED = True  # @param {type:"boolean"}

In [6]:
# C1. 根据 C0 的选项建模；验证检测损失、优化步骤和 checkpoint。
from tempfile import TemporaryDirectory


def make_smoke_model(use_imagenet):
    detector = YOLO(str(CFG_PATH), task='detect')
    if not use_imagenet:
        # 默认路线：保留 YAML 随机初始化，不下载或读取任何 RepViT 权重。
        return detector, 'repvit_m0_9_smoke'

    from argparse import Namespace

    url = (
        'https://github.com/THU-MIG/RepViT/releases/download/v1.0/'
        'repvit_m0_9_distill_450e.pth'
    )
    # 官方训练 checkpoint 用 Namespace 存训练配置；仅对这次官方权重加载放行该类型。
    with torch.serialization.safe_globals([Namespace]):
        checkpoint = torch.hub.load_state_dict_from_url(
            url, map_location='cpu', weights_only=True, progress=True,
        )

    # 模型第 0 层是 RepViT 骨干。键名与官方 features.* 对齐，不导入分类/蒸馏头。
    backbone = detector.model.model[0]
    backbone_weights = {
        key: value for key, value in checkpoint['model'].items()
        if key.startswith('features.')
    }
    # 严格检查键名和形状；加载失败时停止，不悄悄退回随机初始化。
    backbone.load_state_dict(backbone_weights, strict=True)
    print(f'已加载 ImageNet RepViT 骨干：{len(backbone_weights)} 项参数和缓冲区')

    # v8.4.0 从 YAML 启动 train() 会重新建模；通过本地 .pt 把骨干权重交给 trainer。
    # 只重新加载本单元自己生成的文件。训练期间保留该文件，结束后自动清理。
    return detector, 'repvit_m0_9_imagenet_smoke'


use_imagenet = USE_IMAGENET_PRETRAINED
smoke, run_name = make_smoke_model(use_imagenet)
print('初始化方式:', 'ImageNet 骨干 + 随机检测头' if use_imagenet else '全模型随机初始化')
with TemporaryDirectory(prefix='repvit-init-') as temp_dir:
    if use_imagenet:
        init_path = Path(temp_dir) / 'yolo26_repvit_imagenet_init.pt'
        smoke.save(str(init_path))
        smoke = YOLO(str(init_path), task='detect')

    smoke.train(
        data='coco8.yaml', epochs=1, imgsz=320, batch=2, workers=0,
        # 优先使用上面选定的模型初始化，不从其他权重恢复训练进度。
        pretrained=use_imagenet, resume=False, amp=False, plots=False,
        device=0 if torch.cuda.is_available() else 'cpu',
        project='/content/yolo26_repvit_work/runs', name=run_name,
    )

best = Path(smoke.trainer.best)
assert best.is_file(), f'未找到 checkpoint: {best}'
reloaded = YOLO(str(best), task='detect')
assert isinstance(reloaded.model.model[0], RepViT_M0_9)
print('smoke OK；已重新加载:', best)

Downloading: "https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_450e.pth" to /root/.cache/torch/hub/checkpoints/repvit_m0_9_distill_450e.pth


100%|██████████| 21.4M/21.4M [00:00<00:00, 41.2MB/s]


已加载 ImageNet RepViT 骨干：699 项参数和缓冲区
初始化方式: ImageNet 骨干 + 随机检测头
New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tmp/repvit-init-sf029nff/yolo26_repvit_imagenet_i

## D 段：方案 2 — RepViT + 轻量 FPN/PAN + Detect

先完成 A 段、重启会话并运行 B1，再到 D 段。**不用执行 C 段**。无 Neck 的方案 1 及其冒烟代码、原 YAML 和输出记录保持不变；D1 只生成 `yolo26_repvit_light_neck.yaml`。

Neck 中间宽度统一为 **128**：1×1 横向投影 → 两次 nearest 上采样融合（FPN）→ 两次 stride 2 深度卷积下采样融合（PAN）。每次融合采用 `Concat` → `DWConv(3×3)` → `Conv(1×1)`；拼接后是 256 通道，`DWConv` 输入和输出均为 256，所以每个通道单独卷积。YOLO26 的 `Conv` / `DWConv` 自带 BN 和默认 SiLU，不再手写模块或改解析器。

在 Neck 末端再做三次 1×1 输出投影，把 P3/P4/P5 恢复到 **96/192/384** 通道。这使两个方案的 `Detect` 构造参数、内部通道及训练/推理分支保持完全一致。输入给 Neck 的 H/W 应是 32 的倍数；常规 YOLO 数据预处理负责对齐。


In [4]:
# D1. 保留 CFG_PATH（方案 1），单独写入方案 2 的 YAML。
LIGHT_NECK_YAML = '''# YOLO26 + RepViT-M0.9 + lightweight FPN/PAN.
nc: 80
end2end: true
reg_max: 1

backbone:
  - [-1, 1, RepViT_M0_9, []]       # 0: same backbone as the no-neck model

head:
  - [0, 1, Index, [96, 0]]         # 1: F3 (stride 8)
  - [0, 1, Index, [192, 1]]        # 2: F4 (stride 16)
  - [0, 1, Index, [384, 2]]        # 3: F5 (stride 32)
  - [1, 1, Conv, [128, 1, 1]]      # 4: L3
  - [2, 1, Conv, [128, 1, 1]]      # 5: L4
  - [3, 1, Conv, [128, 1, 1]]      # 6: L5

  - [6, 1, nn.Upsample, [null, 2, nearest]]  # 7: L5 -> stride 16
  - [[7, 5], 1, Concat, [1]]       # 8: 128 + 128 = 256
  - [-1, 1, DWConv, [256, 3, 1]]   # 9: groups=256
  - [-1, 1, Conv, [128, 1, 1]]     # 10: T4
  - [10, 1, nn.Upsample, [null, 2, nearest]]  # 11: T4 -> stride 8
  - [[11, 4], 1, Concat, [1]]      # 12: 256 channels
  - [-1, 1, DWConv, [256, 3, 1]]   # 13
  - [-1, 1, Conv, [128, 1, 1]]     # 14: T3

  - [14, 1, DWConv, [128, 3, 2]]   # 15: T3 -> stride 16, groups=128
  - [[15, 10], 1, Concat, [1]]     # 16: 256 channels
  - [-1, 1, DWConv, [256, 3, 1]]   # 17
  - [-1, 1, Conv, [128, 1, 1]]     # 18: fused stride 16
  - [18, 1, DWConv, [128, 3, 2]]   # 19: fused stride 16 -> stride 32
  - [[19, 6], 1, Concat, [1]]      # 20: 256 channels
  - [-1, 1, DWConv, [256, 3, 1]]   # 21
  - [-1, 1, Conv, [128, 1, 1]]     # 22: fused stride 32

  - [14, 1, Conv, [96, 1, 1]]      # 23: P3 (match original Detect input)
  - [18, 1, Conv, [192, 1, 1]]     # 24: P4
  - [22, 1, Conv, [384, 1, 1]]     # 25: P5
  - [[23, 24, 25], 1, Detect, [nc]]  # 26: same Detect as the no-neck model
'''

if not CFG_PATH.is_file():
    raise FileNotFoundError('找不到方案 1 YAML，请先完成 A 段并执行 B1。')
LIGHT_NECK_CFG = CFG_PATH.with_name('yolo26_repvit_light_neck.yaml')
if LIGHT_NECK_CFG.exists() and LIGHT_NECK_CFG.read_text(encoding='utf-8') != LIGHT_NECK_YAML:
    raise RuntimeError(f'不覆盖不同内容的已有配置：{LIGHT_NECK_CFG}')
LIGHT_NECK_CFG.write_text(LIGHT_NECK_YAML, encoding='utf-8')
print('方案 1:', CFG_PATH)
print('方案 2:', LIGHT_NECK_CFG)


方案 1: /content/yolo26_repvit_work/ultralytics/ultralytics/cfg/models/26/yolo26_repvit.yaml
方案 2: /content/yolo26_repvit_work/ultralytics/ultralytics/cfg/models/26/yolo26_repvit_light_neck.yaml


In [5]:
# D2. 构建两个模型，验证解析、分支连接、通道、分辨率、stride 和真实前向。
import torch
from ultralytics import YOLO
from ultralytics.nn.modules import Concat, Conv, DWConv, Detect


def check_light_neck(cfg_no_neck, cfg_light_neck, device):
    plain = YOLO(str(cfg_no_neck), task='detect').model.to(device).eval()
    light = YOLO(str(cfg_light_neck), task='detect').model.to(device).eval()
    base_head, light_head = plain.model[-1], light.model[-1]
    assert type(plain.model[0]) is type(light.model[0]) is RepViT_M0_9
    assert plain.yaml['backbone'] == light.yaml['backbone']
    assert plain.yaml['head'][:3] == light.yaml['head'][:3]  # same Index taps
    assert type(base_head) is type(light_head) is Detect
    assert (base_head.nc, base_head.reg_max, base_head.end2end, base_head.legacy) == (
        light_head.nc, light_head.reg_max, light_head.end2end, light_head.legacy,
    ) == (80, 1, True, True)
    assert {k: v.shape for k, v in base_head.state_dict().items()} == {
        k: v.shape for k, v in light_head.state_dict().items()
    }
    assert plain.stride.tolist() == light.stride.tolist() == [8.0, 16.0, 32.0]
    assert base_head.f == [1, 2, 3] and light_head.f == [23, 24, 25]
    assert all(isinstance(light.model[i], Conv) and light.model[i].conv.kernel_size == (1, 1)
               for i in (4, 5, 6, 10, 14, 18, 22, 23, 24, 25))
    for i, channels, step in [(9,256,1), (13,256,1), (15,128,2), (17,256,1), (19,128,2), (21,256,1)]:
        layer = light.model[i]
        assert isinstance(layer, DWConv)
        conv = layer.conv
        assert conv.groups == conv.in_channels == conv.out_channels == channels
        assert conv.kernel_size == (3, 3) and conv.stride == (step, step)
    assert all(isinstance(light.model[i], Concat) and light.model[i].d == 1 for i in (8, 12, 16, 20))
    assert all(isinstance(light.model[i], torch.nn.Upsample) and light.model[i].mode == 'nearest'
               and light.model[i].scale_factor == 2 for i in (7, 11))

    # 每个条目记录 (通道数, stride)，包括所有 Concat 的输出和两次上下采样。
    expected = {
        1:(96,8), 2:(192,16), 3:(384,32), 4:(128,8), 5:(128,16), 6:(128,32),
        7:(128,16), 8:(256,16), 9:(256,16), 10:(128,16),
        11:(128,8), 12:(256,8), 13:(256,8), 14:(128,8),
        15:(128,16), 16:(256,16), 17:(256,16), 18:(128,16),
        19:(128,32), 20:(256,32), 21:(256,32), 22:(128,32),
        23:(96,8), 24:(192,16), 25:(384,32),
    }
    shapes = {}
    def remember(i):
        return lambda layer, args, out: shapes.update({i: tuple(out.shape)})
    hooks = [light.model[i].register_forward_hook(remember(i)) for i in expected]
    try:
        with torch.inference_mode():
            for h, w in [(640, 640), (384, 640)]:
                shapes.clear()
                x = torch.rand(1, 3, h, w, device=device)
                num_pos = sum((h // s) * (w // s) for s in (8, 16, 32))
                for label, net in [('no neck', plain), ('light neck', light)]:
                    det, raw = net(x)
                    assert det.shape == (1, 300, 6) and torch.isfinite(det).all().item()
                    assert set(raw) == {'one2many', 'one2one'}
                    for branch in raw.values():
                        assert branch['boxes'].shape == (1, 4, num_pos)
                        assert branch['scores'].shape == (1, 80, num_pos)
                        actual = [tuple(f.shape) for f in branch['feats']]
                        assert actual == [(1,c,h//s,w//s) for c,s in ((96,8),(192,16),(384,32))]
                    print(f'{label} {h}x{w}: Detect inputs={actual}; positions={num_pos}')
                for i, (c, s) in expected.items():
                    assert shapes[i] == (1, c, h//s, w//s), (i, shapes[i], c, s)
    finally:
        for hook in hooks:
            hook.remove()
    print('解析、FPN/PAN 全部中间层、Detect 输入及前向检查均通过。')


check_light_neck(CFG_PATH, LIGHT_NECK_CFG, DEVICE)


no neck 640x640: Detect inputs=[(1, 96, 80, 80), (1, 192, 40, 40), (1, 384, 20, 20)]; positions=8400
light neck 640x640: Detect inputs=[(1, 96, 80, 80), (1, 192, 40, 40), (1, 384, 20, 20)]; positions=8400
no neck 384x640: Detect inputs=[(1, 96, 48, 80), (1, 192, 24, 40), (1, 384, 12, 20)]; positions=5040
light neck 384x640: Detect inputs=[(1, 96, 48, 80), (1, 192, 24, 40), (1, 384, 12, 20)]; positions=5040
解析、FPN/PAN 全部中间层、Detect 输入及前向检查均通过。


In [6]:
# D3（可选）. 同一设备、同一输入、FP32、batch=1，比较参数、GFLOPs、模型前向延迟。
# eval（不 fuse），含 Detect 后处理；不含读图、预处理、传输。GFLOPs 为估算。
from statistics import median
from time import perf_counter
from ultralytics.utils.torch_utils import get_flops

PROFILE_IMGSZ = 640
x_profile = torch.zeros(1, 3, PROFILE_IMGSZ, PROFILE_IMGSZ, device=DEVICE)


def sync_device():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)


rows = []
for label, cfg in [('no neck', CFG_PATH), ('light neck', LIGHT_NECK_CFG)]:
    net_profile = YOLO(str(cfg), task='detect').model.to(DEVICE).float().eval()
    params = sum(p.numel() for p in net_profile.parameters())
    gflops = get_flops(net_profile, imgsz=PROFILE_IMGSZ)
    seconds = []
    with torch.inference_mode():
        for _ in range(10):
            net_profile(x_profile)
        sync_device()
        for _ in range(30):
            start = perf_counter()
            net_profile(x_profile)
            sync_device()
            seconds.append(perf_counter() - start)
    rows.append((label, params, gflops, 1e3 * median(seconds)))
    del net_profile

del x_profile
for label, p, g, ms in rows:
    print(f'{label:10s} | params={p:,} | GFLOPs={f"{g:.2f}" if g else "N/A"} | median={ms:.2f} ms')
print(f'Neck 参数增量：{rows[1][1] - rows[0][1]:,}')


no neck    | params=6,748,120 | GFLOPs=21.88 | median=15.73 ms
light neck | params=7,068,440 | GFLOPs=23.19 | median=17.60 ms
Neck 参数增量：320,320


## E 段（可选）：两套配置公平对照训练

两个模型使用**完全相同的初始骨干和 Detect 权重**（不仅是相同结构），以及同一份数据、输入尺寸和训练参数。只有新增 Neck 的权重是该方案独有。默认两边都不用预训练；如果开启开关，两边都迁移同一份官方 RepViT-M0.9 权重，不迁移分类头。

默认与 C 段一样用 COCO8 跑一轮，只验证训练链路。比较检测精度时，请把 `COMPARE_DATA` 改为同一正式检测数据 YAML，并设置适合的训练轮数和输入尺寸；两边会使用完全相同的设置。不要把 COCO8 的 mAP 作为结论。用 D3 比较未融合的前向延迟；如测部署速度，两个模型应采用同一导出及推理设置。


In [7]:
# E1. 运行这一格会训练两次，结果分别保存；不影响 C 段原有的冒烟实验。
from argparse import Namespace
from tempfile import TemporaryDirectory

COMPARE_DATA = 'coco8.yaml'
COMPARE_EPOCHS = 1
COMPARE_IMGSZ = 320
COMPARE_BATCH = 2
COMPARE_USE_IMAGENET = False  # @param {type:"boolean"}

compare_train_args = dict(
    data=COMPARE_DATA, epochs=COMPARE_EPOCHS, imgsz=COMPARE_IMGSZ, batch=COMPARE_BATCH,
    workers=0, device=0 if torch.cuda.is_available() else 'cpu', amp=False, plots=False,
    seed=0, deterministic=True, resume=False, pretrained=COMPARE_USE_IMAGENET,
    project='/content/yolo26_repvit_work/runs/neck_compare',
)

plain_compare = YOLO(str(CFG_PATH), task='detect')
light_compare = YOLO(str(LIGHT_NECK_CFG), task='detect')
if COMPARE_USE_IMAGENET:
    url = ('https://github.com/THU-MIG/RepViT/releases/download/v1.0/'
           'repvit_m0_9_distill_450e.pth')
    with torch.serialization.safe_globals([Namespace]):
        ckpt = torch.hub.load_state_dict_from_url(url, map_location='cpu', weights_only=True)
    plain_compare.model.model[0].load_state_dict(
        {k:v for k,v in ckpt['model'].items() if k.startswith('features.')}, strict=True,
    )
    del ckpt
# 复制所有骨干和 Detect 参数/缓冲区，消除建模顺序造成的随机初始化差异。
for i in (0, -1):
    light_compare.model.model[i].load_state_dict(plain_compare.model.model[i].state_dict(), strict=True)

comparison = {}
with TemporaryDirectory(prefix='repvit-neck-compare-') as tmp:
    recipes = [('no_neck', plain_compare), ('light_neck', light_compare)]
    for label, detector in recipes:
        # 使用 .pt 入口，确保 YOLO26 trainer 重建模型时保留上面复制的初始化。
        init = Path(tmp) / f'{label}.pt'
        detector.save(str(init))
        run = YOLO(str(init), task='detect')
        result = run.train(**compare_train_args, name=label + ('_imagenet' if COMPARE_USE_IMAGENET else '_random'))
        comparison[label] = (Path(run.trainer.best), dict(result.results_dict))
        assert comparison[label][0].is_file()
        del run

for label, (best_path, metrics) in comparison.items():
    print(label, best_path)
    for metric in ('metrics/mAP50(B)', 'metrics/mAP50-95(B)'):
        print(f'  {metric}: {metrics.get(metric, "N/A")}')


New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tmp/repvit-neck-compare-py8091t5/no_neck.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=no_neck_random, 